In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from dataclasses import dataclass
from collections import deque
from typing import List
from math import ceil, log, exp
import heapq
import copy
from __future__ import annotations

In [ ]:
@dataclass
class Event:
    time: float
    event_type: str
    obj: EV | int # It's either an EV or an integer (which is the last Event)
    valid: bool = True
    
    def __lt__(self, other: EV):
        return self.time < other.time


@dataclass
class EV:
    id: int
    c_b: float       # Battery capacity (kWh)
    s_i: float    # Initial SoC (0.0 - 1.0)
    s_f: float   # Target SoC (0.0 - 1.0)
    arrival_time: float
    s_th: float = 0.6
    c_rate: float = 1 # C-rate: p_m = C_b * C_rate
    service_start_time: float = None 
    pile: ChargingPile = None # Reference to assigned charging pile
    pile_tracker: ChargingPile = None
    nozzle_id: int = None # ID of the nozzle when EV is connected to the pile
    energy_received: float = 0 # Energy received (kWh)
    energy_received_2: float = 0 # Energy received (kWh)
    departure_time: float = float('inf')

    def __post_init__(self):
        self.p_req_max = self.c_b * self.c_rate    # Peak charging request rate (kW)
        self.tan_B = self.p_req_max / (1 - self.s_th) # Decay tangent
        self.s_current = self.s_i               # Current SoC
        self.t_th = self.c_b / self.p_req_max * (self.s_th - self.s_i)    # Time corresponding to s_thresh
        self.energy_needed = (self.s_f - self.s_i) * self.c_b # Energy required (kWh)

    def assign_to_pile(self, pile: ChargingPile):
        pile.assign_ev(self)
    
    def remove_from_pile(self, pile: ChargingPile):
        pile.remove_ev(self)


    @property
    def current_time(self) -> float:
        if self.pile is None:
            return float('inf')
        return self.pile.station.current_time

    @property
    def next_time(self) -> float:
        """
        This is the next global event (based on other EVs, Arrivals, ...)
        """
        if self.pile is None:
            return float('inf')
        return self.pile.station.next_time
    
    @property
    def n_bricks(self) -> int:
        if self.pile is None:
            return None
        return self.pile.EV_bricks[self.nozzle_id]

    @property
    def p_req(self) -> float:
        """
        The BMS power request based on the current SoC
        """
        # # This is handled in the ChargingStation logic right now:
        # # Valid only if EV is plugged and charging
        # if self.pile is None or self.p_act == 0:
        #     return None
        return self.tan_B * (1 - max(self.s_th, self.s_current))

    @property
    def p_act(self) -> float:
        # The actual power drawn from the charging pile
        return min(self.p_req, self.n_bricks * self.pile.p_brick)

    @property
    def s_taper(self) -> float:
        """
        Computes taper SoC based on the power received by the vehicle
        if p_act = p_req_max then s_taper = s_th
        """
        return 1 - self.p_act / self.tan_B
    
    @property
    def t_taper(self) -> float:
        return self.c_b / self.p_act * (self.s_taper - self.s_current) + self.current_time
    
    @property
    def next_state(self) -> tuple:
        """
        Next SoC of interest: if we are in the constant zone of charging curve
        next event is hitting the taper on the charging curve
        elif next SoC is under utilizing the power brick
        else it is the final SoC
        """
        if self.pile.isOverloaded and self.n_bricks > 1:
            # Power corresponding to underutiizing a power brick
            p_thresh = (self.n_bricks - 1 + self.pile.brickCheck_thresh) * self.pile.p_brick
            # Power corresponding to freeing up a power brick
            p_thresh_2 = (self.n_bricks - 1) * self.pile.p_brick
            # If self.p_act < p_thresh, next state is freeing up that power brick
            # else it is underutilizing that brick
            p_next = max(min(p_thresh, self.p_act), min(p_thresh_2, self.p_act))
            return (1 - p_next / self.tan_B, "ChargeChange")
        else:
            return (self.s_f, "Departure")

    @property
    def s_next_candidate(self) -> float:
        return self.next_state[0]

    @property
    def t_next_candidate(self) -> float:
        """
        Computes the time of next charging event for this EV; 
        """       
        linear_term = min(self.s_taper, self.s_next_candidate) - min(self.s_taper, self.s_current)
        log_term = - (1 - self.s_taper) * log((1 - max(self.s_taper, self.s_next_candidate)) / (1 - max(self.s_taper, self.s_current)))
        return self.c_b / self.p_act * (linear_term + log_term) + self.current_time

    @property
    def s_next(self) -> float:
        delta_t = self.next_time - self.current_time
        k_tmp = delta_t * self.p_act / self.c_b
        # Constant part of P-S curve
        if self.current_time < self.next_time <= self.t_taper:
            return k_tmp + self.s_current
        # Starting at the constant part - going in the linear decay
        elif self.current_time < self.t_taper < self.next_time:
            exp_term = exp((self.s_taper - self.s_current - k_tmp) / (1 - self.s_taper))
            return 1 - (1 - self.s_taper) * exp_term
        # Linear decay part
        elif self.t_taper <= self.current_time < self.next_time:
            return 1 - (1 - self.s_current) * exp(-k_tmp / (1 - self.s_taper))
        else:
            raise ValueError("Something is wrong with s_next calculation!")
    
    @property
    def deltaE_power(self) -> float:
        linear_term = min(self.t_taper, self.next_time) - min(self.t_taper, self.current_time)
        expo_next_time = exp(-self.tan_B/self.c_b * (max(self.next_time, self.t_taper) - self.t_taper))
        expo_current_time = exp(-self.tan_B/self.c_b * (max(self.current_time, self.t_taper) - self.t_taper))
        expo_term = -self.c_b/self.tan_B * (expo_next_time - expo_current_time)
        return self.p_act * (linear_term + expo_term)
    
    @property
    def deltaE_SoC(self):
        return self.c_b * (self.s_next - self.s_current)

    @property
    def SoC_check(self) -> bool:
        return np.isclose(self.s_current, self.s_f, rtol=1e-6, atol=1e-9)

    @property
    def energy_received_check(self) -> bool:
        return np.isclose(self.energy_received, self.energy_needed, rtol=1e-6, atol=1e-9)

    @property
    def energy_received_check_2(self) -> bool:
        return np.isclose(self.energy_received, self.energy_received_2, rtol=1e-6, atol=1e-9)


@dataclass
class ChargingPile:
    id: int
    n: int # Number of nozzles for this pile
    num_bricks: int # Max number of power bricks for this pile
    p_brick: float # kW per power brick
    brickCheck_thresh: float = 0.2 # The threshold to check when an EV is utilizing less than this ratio of its last brick, considering assigning that brick to another EV where they might beter benefit from that
    station: ChargingStationEnv = None

    def __post_init__(self):
        if self.num_bricks < self.n:
            raise ValueError(f"num_bricks: {self.num_bricks} is less than the number of nozzles: {self.n}")
        self.power_supp: float = self.num_bricks * self.p_brick
        self.EVs: List[EV] = [] # EVs assigned to this charging pile; each EV is assigned to a nozzle
        self.finished_EVs: List[EV] = [] # EVs finished charging and left this pile
        self.EV_bricks: List[int] = [] # Number of power bricks assigned to each EV

    @property
    def current_time(self) -> float:
        if self.station is None:
            return float('inf')
        return self.station.current_time

    @property
    def next_time(self) -> float:
        """
        This is the next global event (based on other EVs, Arrivals, ...)
        """
        if self.station is None:
            return float('inf')
        return self.station.next_time

    def assign_ev(self, ev: EV):
        if ev.pile is not None:
            raise ValueError("EV already assigned to a pile.")
        if len(self.EVs) > self.n:
            raise ValueError("Charging Pile is full.")
        ev.pile = self # Link back
        ev.pile_tracker = self
        ev.nozzle_id = len(self.EVs)
        ev.service_start_time = self.next_time # Log Arrival
        self.EVs.append(ev)
        #self.EV_bricks.append(0)
        self.distributePower()
    
    def remove_ev(self, ev: EV):
        if ev.pile is None:
            raise ValueError("EV is not assigned to any pile.")
        if ev.pile != self:
            raise ValueError("EV is not assigned to this pile.")
        idx = self.EVs.index(ev)
        self.EVs.pop(idx) # Remove EV
        #self.EV_bricks[idx] -= 1 # Free up its assigned bricks
        self.finished_EVs.append(ev)
        ev.pile = None
        ev.nozzle_id = None
        ev.departure_time = self.next_time # Log Departure
        self.station.finished_evs.append(ev)
        self.distributePower()

    @property
    def isActive(self) -> bool: # If any EV is connected to this pile
        if len(self.EVs) > 0:
            return True
        return False
    
    @property
    def isOverloaded(self) -> bool:     # If power req is more than demand (in terms of power bricks)
        if sum(ceil(ev.p_req / self.p_brick) for ev in self.EVs) > self.num_bricks:
            return True
        return False
    
    @property
    def power_reqs(self):
        # Keeps track of total power request of each EV put to the charging pile
        return [ev.p_req for ev in self.EVs]
    
    @property
    def remaining_reqs(self):
        # Keep tracks of unfulfilled power request of each ev
        return [ev.p_req - ev.n_bricks * self.p_brick for ev in self.EVs]

    def distributePower(self):
        """
        Takes a power requests, assigns power bricks based on the weighted ratio of the 
        power requests to EVs 
        """
        if self.isOverloaded:
            total_req = sum(self.power_reqs)
            self.EV_bricks = [max(1, (ev.p_req / total_req * min(total_req, self.power_supp) // self.p_brick)) for ev in self.EVs] # At least assign one brick to each request
            self.microDistribute()
        else:
            self.EV_bricks = [ceil(ev.p_req / self.p_brick) for ev in self.EVs]

    def microDistribute(self, ev: EV | None = None):
        """
        Do a microDistribution, which assigns any remaining power brick to
        the vehicle most needing it.
        """
        if ev: #Unassign brick from the EV underultilizing it
            self.EV_bricks[ev.nozzle_id] -= 1
        while self.num_bricks > sum(self.EV_bricks):
            sorted_args = deque(np.argsort(self.remaining_reqs)[::-1])
            self.EV_bricks[sorted_args[0]] += 1        


class ChargingStationEnv(gym.Env):
    def __init__(self, n=4, n_nozzle=2, n_brick=5, p_brick=25, c_q=10, lam=5.0, max_time=1440):
        super().__init__()
        
        # System Parameters
        self.n = n          # Number of piles
        self.n_nozzle = n_nozzle
        self.n_brick = n_brick
        self.p_brick = p_brick
        self.c_q = c_q      # Queue capacity
        self.lam = lam      # Arrival rate (mean minutes); Number of arrivals in any interval of length t is Poisson(t / lambda)
        # and each inter-arrival time is Exp(1 / lambda)
        self.max_time = max_time # 24 hours in minutes
        
        # EV Distribution Parameters
        self.c_b_options = [50.0, 100.0, 150.0]

        # Action Space: 0 to n-1 (assign to pile), n (Wait / Advance Time)
        self.action_space = spaces.Discrete(self.n + 1) 

        # State tracking
        self.current_time: float = 0
        self.next_time: float = 0
        self.event_type: str = None
        self.next_ev: EV | float = None
        self.queue: List[EV] = []
        self.piles: List[ChargingPile] = [ChargingPile(_, n_nozzle, n_brick, p_brick) for _ in range(self.n)]
        self.arrival_schedule: List[float] = []
        self.arrived_evs: List[EV] = []
        self.finished_evs: List[EV] = [] # For Metric Collection
        self.dropped_cars: int = 0
        self.terminate: bool = False
        # Track history of L (Cars in system) for Little's Law
        self.history_L: List[int] = []
        # Track history of Cars in Queue
        self.history_Q: List[int] = []
        # Track history of event times for validation and metric calculation 
        self.events_sch: List[float] = []
        # Track history of delta T, Element i of this list is the time system had 
        # self.history_L[i] and self.history_Q[i] and ...
        self.events_delta_t: List[float] = []
        # Utilization and Energy Tracking of Piles and Nozzles
        self.pile_active_minutes = np.zeros(self.n)
        self.nozzle_active_minutes = np.zeros((self.n, self.n_nozzle))
        self.pile_energy_sold = np.zeros(self.n)
        self.nozzle_energy_sold = np.zeros((self.n, self.n_nozzle))
        # RNG (initialized in reset)
        self.rng_arrival: np.random.Generator | None = None
        self.rng_choice: np.random.Generator | None = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng_arrival = np.random.default_rng(seed)
            self.rng_choice = np.random.default_rng(seed+240)
            
        self.current_time = 0
        self.terminated = False
        self.queue = []
        self.piles = [ChargingPile(_, self.n_nozzle, self.n_brick, self.p_brick) for _ in range(self.n)]
        self.arrived_evs = []
        self.finished_evs = []
        self.dropped_cars = 0
        self.history_L = []
        self.history_Q = []
        self.events_sch = []
        self.events_delta_t = []

        self.pile_active_minutes = np.zeros(self.n)
        self.nozzle_active_minutes = np.zeros((self.n, self.n_nozzle))
        self.pile_energy_sold = np.zeros(self.n)
        self.nozzle_energy_sold = np.zeros((self.n, self.n_nozzle))
        
        # Assign each pile to the station
        self._assign_pile()

        # Pre-generate the entire day's arrivals
        self._pre_generate_arrivals()
        
        # Find the next event
        self._load_next_event()
        # self.next_time = we have it 
        # self.current_time = 0
        self._update_metrics()
        # current_L = 0
        # self.history_L = [0]
        # self.current_time = [0]

        self._update_state()
        # self.event_type == "Arrival"
        # self.queue = [ev]
        # self.current_time = self.next_time
        
        return self._get_obs(), {}

    def _assign_pile(self):
        for pile in self.piles:
            pile.station = self # Link back

    def _pre_generate_arrivals(self):
        """Generates all EV arrivals for the simulation duration
        and stores them as min-heap."""
        # First sample inter-arrival times from the exp distribution
        arrivals = self.rng_arrival.exponential(self.lam, int(self.max_time / self.lam * 1.5)).cumsum()
        arrivals = arrivals[arrivals <= self.max_time].tolist()
        ev_counter = 0
        
        for t in arrivals:
            ev_counter += 1
            c_b = self.rng_arrival.choice(self.c_b_options)
            s_init = self.rng_arrival.uniform(0.10, 0.30)
            s_final = self.rng_arrival.uniform(0.70, 0.90)
            s_thresh = 0.6
            
            new_ev = EV(ev_counter, c_b, s_init, s_final, t, s_th=s_thresh)
            heapq.heappush(self.arrival_schedule, Event(new_ev.arrival_time, "Arrival", new_ev))
        
        # Push the simulation end time to the heap
        heapq.heappush(self.arrival_schedule, Event(self.max_time, "Sim Over", self.max_time))

    def _load_next_event(self):
        """Finds the next event by combining arrival events with departure
        and ChargeChange events of EVs already plugged in."""
        tmp_event_schedules = copy.deepcopy(self.arrival_schedule)
        for pile in self.piles:
            for ev in pile.EVs:
                heapq.heappush(tmp_event_schedules, Event(ev.t_next_candidate, ev.next_state[-1], ev))
        # Pop the next event, its corresponding EV, and the time
        next_one = heapq.heappop(tmp_event_schedules)
        self.next_ev = next_one.obj
        self.next_time = next_one.time
        self.event_type = next_one.event_type

    def _update_metrics(self):
        """
        Update the metrics of piles, nozzles, EVs
        """
        current_L = len(self.queue) + sum(len(pile.EVs) for pile in self.piles)
        self.history_L.append(current_L)
        self.history_Q.append(len(self.queue))
        self.events_sch.append(self.current_time)
        delta_t = self.next_time - self.current_time
        self.events_delta_t.append(delta_t)

        for pile_idx, pile in enumerate(self.piles):
            if pile.isActive:
                # Pile Tracking
                self.pile_active_minutes[pile_idx] += delta_t
                # Iterate over EVs and their 'nozzle' index
                for nozzle_idx, ev in enumerate(pile.EVs):
                    # Nozzle Tracking
                    self.nozzle_active_minutes[pile_idx][nozzle_idx] += delta_t
                    self.nozzle_energy_sold[pile_idx][nozzle_idx] += ev.deltaE_power
                    self.pile_energy_sold[pile_idx] += ev.deltaE_power # Aggregate for pile
                    # EV Tracking
                    ev.energy_received += ev.deltaE_power
                    ev.energy_received_2 += ev.deltaE_SoC

    def _update_state(self):
        """
        Update the state of vehicles
        """
        for pile in self.piles:
            for ev in pile.EVs:
                ev.s_current = ev.s_next

        # If Arrival, append that vehicle to the end of the queue
        if self.event_type == "Arrival":
            # Remove that arrival event from the arrival schedule as well
            heapq.heappop(self.arrival_schedule)
            self.arrived_evs.append(self.next_ev)
            if len(self.queue) < self.c_q:
                self.queue.append(self.next_ev)
            else:
                self.dropped_cars += 1 # Metric: Queue full
        # If ChargeChange, do a microDistribution
        elif self.event_type == "ChargeChange":
            self.next_ev.pile.microDistribute(self.next_ev)
        # If Departure, a distributePower() is done inside the remove_from_pile method 
        elif self.event_type == "Departure":
            self.next_ev.remove_from_pile(self.next_ev.pile)
        # If Simulation Over, set self.terminated to True
        elif self.event_type == "Sim Over":
            self.terminated = True
        else:
            raise ValueError("Something is off ...")
        
        self.current_time = self.next_time

    def _get_obs(self):
        return {
            "time": self.current_time,
            "queue_len": len(self.queue),
            "pile_occupancy": [len(pile.EVs) for pile in self.piles],
            "piles": [pile for pile in self.piles],
            "head_ev": self.queue[0] if self.queue else None
        }

    def step(self, action):
        reward = 0
        
        # --- CASE 1: Instant Assignment (No time passes) ---
        if action < self.n:
            if len(self.queue) > 0:
                target_pile = self.piles[action]
                if len(target_pile.EVs) < target_pile.n: # If there are empty nozzles in the target pile
                    ev = self.queue.pop(0)
                    target_pile.assign_ev(ev) # Assign ev to the target pile
                    target_pile.distributePower() # Compute the power dist for each EV plugged in 
                    reward += 1
                else:
                    reward -= 1 # Penalty for illegal move (pile full)
            else:
                reward -= 1 # Penalty for trying to assign from empty queue
                
            # Return immediately without advancing time
            return self._get_obs(), reward, False, False, {}

        # --- CASE 2: Wait / Advance Time (Action == n) ---
        # Load the next event time and type ("Arrival", "ChargeChange", "Departure", "Sim Over")
        self._load_next_event()
        # self.next_time = we have it
        # Update metrics BEFORE updating state and advancing time
        self._update_metrics()
        # Current_L = 1
        # Update state and advance time
        self._update_state()
        if self.event_type == "Departure":
            reward += 10 # Reward for finishing
        # current_t ------x update metric | update s, compute 
        return self._get_obs(), reward, self.terminated, False, {}

In [3]:
def fifo_heuristic_agent(obs, n, rng):
    """
    Revised logic: If there's a car in queue and space in a pile, ASSIGN.
    Otherwise, ADVANCE TIME (action = n).
    """
    try:
        queue_len = obs["queue_len"]
        # If queue is empty, we must advance time.
        if queue_len == 0:
            return n
            
        nozzle_capacity = [pile.n - len(pile.EVs) for pile in obs["piles"]]
        
        # If queue has cars but all piles are full, we must advance time.
        if sum(nozzle_capacity) == 0:
            return n 
            
        # Standard FIFO logic: pick the least occupied pile
        max_cap = max(nozzle_capacity)
        best_piles = [idx for idx, nozzle in enumerate(nozzle_capacity) if nozzle == max_cap]
        
        return rng.choice(best_piles)   
    except Exception as e:
        print(f"Error: {e}")
        print(f"Av Nozzles: {nozzle_capacity}")
        print(f"min occ: {max_cap}")
        print(best_piles)
        raise

In [4]:
def report_metrics(env: ChargingStationEnv):
    """Calculates and prints standard queueing metrics."""
    served_evs = env.finished_evs
    arrived_evs = env.arrived_evs
    num_served = len(served_evs)
    num_arrived = len(arrived_evs)

    if num_served == 0:
        print("No EVs were served.")
        return

    wait_times = [ev.service_start_time - ev.arrival_time for ev in served_evs]
    service_times = [ev.departure_time - ev.service_start_time for ev in served_evs]
    total_time_in_sys = [ev.departure_time - ev.arrival_time for ev in served_evs]

    print(f"--- 24-Hour Simulation Results ---")
    print(f"Mean Inter-arrival Rate: {env.lam} min")
    print(f"Total Arrivals Simulated: {num_arrived}")
    print(f"Total Cars Served:        {num_served}")
    print(f"Cars Dropped (Queue Full): {env.dropped_cars}")
    print(
        f"Cars Remaining in System: {sum(len(p.EVs) for p in env.piles) + len(env.queue)}"
    )
    print("------------------------------------")
    print(f"Avg Wait Time in Queue ($W_q$): {np.mean(wait_times):.2f} mins")
    print(f"Avg Service/Charge Time:      {np.mean(service_times):.2f} mins")
    print(f"Avg Total Time in Sys ($W$):    {np.mean(total_time_in_sys):.2f} mins")

    # Calculate Utilization Rates (%)
    total_minutes = env.max_time
    pile_utilization = (env.pile_active_minutes / total_minutes) * 100
    nozzle_utilization = (env.nozzle_active_minutes / total_minutes) * 100

    print("\n--- PILE & NOZZLE METRICS ---")
    # Generate dynamic header
    nozzle_headers = " | ".join([f"Nozzle {i} (Util/kWh)" for i in range(env.n_nozzle)])
    print(
        f"{'Pile':<6} | {'Pile Util (%)':<15} | {'Pile Energy (kWh)':<18} | {nozzle_headers}"
    )
    print("-" * (88 + 23 * (env.n_nozzle - 2)))  # Adjust separator length

    for pile_idx in range(env.n):
        nozzle_strs = []
        for nozzle_idx in range(env.n_nozzle):
            # Format Nozzle 0 stats
            n_util = nozzle_utilization[pile_idx][nozzle_idx]
            n_energy = env.nozzle_energy_sold[pile_idx][nozzle_idx]
            nozzle_strs.append(f"{n_util:.1f}% / {n_energy:.1f}")

        # Join all nozzle strings with separator
        nozzle_columns = " | ".join([f"{s:<20}" for s in nozzle_strs])

        print(
            f"P-{pile_idx:<4} | {pile_utilization[pile_idx]:<15.1f} | {env.pile_energy_sold[pile_idx]:<18.1f} | {nozzle_columns}"
        )

    print("-" * (88 + 23 * (env.n_nozzle - 2)))  # Adjust separator length

    total_energy = np.sum(env.pile_energy_sold)
    print(f"Total Energy Sold: {total_energy:.2f} kWh")


def perform_sanity_checks(env):
    """
    Checks the validity of the simulation using Little's Law
    and Energy Conservation principles.
    """
    print("\n--- ANALYTICAL SANITY CHECKS ---")

    # 1. LITTLE'S LAW CHECK: L = lambda_eff * W

    if not env.history_L:
        print("Error: No L history tracked. Cannot perform Little's Law check.")
        return

    # Calculate Avg L (Time-average number of cars in system)
    avg_L_simulated = np.sum([l * delta_t for (l, delta_t) in zip(env.history_L, env.events_delta_t)]) / env.events_sch[-1]

    # Calculate effective lambda (Throughput)
    total_time_mins = env.max_time
    total_served = len(env.finished_evs)
    lambda_eff = total_served / total_time_mins

    # Average time in system (W)
    if total_served > 0:
        total_time_in_sys = [
            ev.departure_time - ev.arrival_time for ev in env.finished_evs
        ]
        avg_W_simulated = np.mean(total_time_in_sys)
    else:
        avg_W_simulated = 0

    L_theoretical = lambda_eff * avg_W_simulated

    print(f"[Little's Law Check]")
    print(f"  Simulated Avg L (from history):   {avg_L_simulated:.4f} cars")
    print(f"  Theoretical L (Throughput * W):   {L_theoretical:.4f} cars")

    # The discrepancy should be small (< 0.5 cars usually, depending on start/end transients)
    diff = abs(avg_L_simulated - L_theoretical)
    print(f"  Difference:                       {diff:.4f}")

    if diff < 0.2:
        print("  >> PASS: Little's Law holds.")
    else:
        print(
            "  >> WARNING: Discrepancy detected (normal if system hasn't reached steady state)."
        )

    # 2. ENERGY CONSERVATION CHECK
    # Energy Needed by Served Cars vs Energy Sold by Piles
    finished_evs = [ev.c_b * (ev.s_f - ev.s_i) for ev in env.finished_evs]
    serving_evs = [ev.c_b * (ev.s_current - ev.s_i) for p in env.piles for ev in p.EVs]
    total_energy_needed = sum(finished_evs + serving_evs)
    total_energy_sold = np.sum(env.pile_energy_sold)

    print(f"\n[Energy Conservation Check]")
    print(f"  Total Energy Demand (Served EVs): {total_energy_needed:.2f} kWh")
    print(f"  Total Energy Sold (Piles):        {total_energy_sold:.2f} kWh")

    # Allow small floating point error
    if abs(total_energy_needed - total_energy_sold) < 1.0:
        print("  >> PASS: Energy In = Energy Out.")
    else:
        print("  >> FAIL: Energy conservation violated.")

In [5]:
# --- RUN THE SIMULATION ---
# n: Number of piles, n_nozzle: Nozzles per pile, n_brick: Max power bricks per pile, p_brick: kW per brick, c_q: Queue capacity
env = ChargingStationEnv(
    n=3, n_nozzle=1, n_brick=5, p_brick=50, c_q=100, lam=3, max_time=1440
)
obs, _ = env.reset(seed=41)  # Seeding for reproducibility

terminated = False
while not terminated:
    action = fifo_heuristic_agent(obs, env.n, env.rng_choice)
    obs, reward, terminated, _, info = env.step(action)

In [6]:
report_metrics(env)

--- 24-Hour Simulation Results ---
Mean Inter-arrival Rate: 3 min
Total Arrivals Simulated: 522
Total Cars Served:        521
Cars Dropped (Queue Full): 0
Cars Remaining in System: 1
------------------------------------
Avg Wait Time in Queue ($W_q$): 0.00 mins
Avg Service/Charge Time:      0.69 mins
Avg Total Time in Sys ($W$):    0.69 mins

--- PILE & NOZZLE METRICS ---
Pile   | Pile Util (%)   | Pile Energy (kWh)  | Nozzle 0 (Util/kWh)
-----------------------------------------------------------------
P-0    | 8.1             | 9948.4             | 8.1% / 9948.4       
P-1    | 8.4             | 10536.9            | 8.4% / 10536.9      
P-2    | 8.6             | 10921.1            | 8.6% / 10921.1      
-----------------------------------------------------------------
Total Energy Sold: 31406.37 kWh


In [13]:
perform_sanity_checks(env)


--- ANALYTICAL SANITY CHECKS ---
[Little's Law Check]
  Simulated Avg L (from history):   1.4545 cars
  Theoretical L (Throughput * W):   1.4420 cars
  Difference:                       0.0125
  >> PASS: Little's Law holds.

[Energy Conservation Check]
  Total Energy Demand (Served EVs): 17399.94 kWh
  Total Energy Sold (Piles):        17399.94 kWh
  >> PASS: Energy In = Energy Out.
